# Segmentation

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
import keras
import os
from keras import backend as K
from keras import layers
from PIL import Image

In [ ]:
# from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy("float16")

https://huggingface.co/datasets/jonathan-roberts1/Ships-In-Satellite-Imagery

https://huggingface.co/datasets/IGNF/PASTIS-HD

https://huggingface.co/datasets/UrbanSyn/UrbanSyn#urbansyn-dataset

LujianYao/HSSDataset

In [ ]:
# from google.colab import drive
# import os

# # Mount Google Drive
# drive.mount('/content/drive')

# # Define the path in Google Drive where you want to clone the dataset
# drive_path = '/content/drive/MyDrive/HSSDataset'

# # Create the directory if it doesn't exist
# if not os.path.exists(drive_path):
#     os.makedirs(drive_path)
#     print(f"Directory '{drive_path}' created.")

#     # Change the current working directory to the drive path
#     %cd {drive_path}

#     # Clone the dataset using git lfs
#     !git lfs install
#     # Ensure git lfs is installed (already done in a previous cell: pERe9L2nEHLy)
#     !git clone https://huggingface.co/datasets/LujianYao/HSSDataset .

#     print(f"Dataset cloned into '{drive_path}'")
# else:
#     print(f"Directory '{drive_path}' already exists. Skipping creation.")



Mounted at /content/drive
Directory '/content/drive/MyDrive/HSSDataset' created.
/content/drive/MyDrive/HSSDataset
Cloning into '.'...
remote: Enumerating objects: 5489, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 5489 (delta 2), reused 0 (delta 0), pack-reused 5483 (from 1)
Receiving objects: 100% (5489/5489), 791.06 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Updating files: 100% (5452/5452), done.
Filtering content: 100% (5441/5441), 3.64 GiB | 12.34 MiB/s, done.
fatal: cannot exec '/content/drive/MyDrive/HSSDataset/.git/hooks/post-checkout': Permission denied
Dataset cloned into '/content/drive/MyDrive/HSSDataset'


In [ ]:
# !git lfs install
# !git clone https://huggingface.co/datasets/LujianYao/HSSDataset

In [6]:
arr = np.load("HSSDataset_organized/train/images/10_frame_0001.npy")
arr.shape

(216, 408, 25)

In [35]:
arr = cv2.imread("HSSDataset_organized/train/masks/10_frame_0001.png")
arr.shape

(216, 408, 3)

In [9]:
arr = np.load("HSSDataset_organized/test/images/11_frame_0433.npy")
arr.shape

(216, 408, 25)

In [10]:
len(os.listdir("HSSDataset_organized"))

3

In [ ]:
# dataset_root = "/content/drive/MyDrive/HSSDataset/HSS_VOC"
# hsi_data_dir = "npy_multichannel"
# splits = ['split/train.txt', 'split/val.txt', 'split/test.txt']
# total_files = 0

# for split in splits:
#     with open(os.path.join(dataset_root, split), 'r') as f:
#         # count number of files in each set
#         files = f.readlines()
#         print(f"Files in {split.rsplit('/', 1)[-1]}", n_files:=len(files))
#         total_files += n_files

#         # check existance of all files
#         for file in files:
#             file_path = os.path.join(dataset_root, hsi_data_dir, file.strip() + '.npy')
#             if not os.path.exists(file_path):
#                 print(f"Warning: File {file_path} does not exist")

#             # check dimension mismatch
#             shape = None
#             arr = np.load(file_path)
#             if shape is None:
#                 shape = arr.shape
#             else:
#                 if shape != arr.shape:
#                     print(f"Warning: File {file_path} has different shape")

# print(f"Total number of files: {total_files}")

Files in train.txt 604
Files in val.txt 201
Files in test.txt 202
Total number of files: 1007


In [ ]:
# import os
# import shutil

# data_root = "/content/drive/MyDrive/HSSDataset/HSS_VOC"
# hsi_data_dir = "npy_multichannel"
# mask_data_dir = "Mask"
# split_dir = "split"
# destination_root = "/content/HSSDataset_Splitted"

# splits = {
#     "train": "train.txt",
#     "val": "val.txt",
#     "test": "test.txt"
# }

# print("Starting dataset organization...")

# for split_name, split_file in splits.items():
#     split_filepath = os.path.join(dataset_root, split_dir, split_file)

#     dest_image_dir = os.path.join(destination_root, split_name, "images")
#     dest_mask_dir = os.path.join(destination_root, split_name, "masks")

#     os.makedirs(dest_image_dir, exist_ok=True)
#     os.makedirs(dest_mask_dir, exist_ok=True)

#     print(f"Processing {split_name} split using {split_filepath}")

#     with open(split_filepath, 'r') as f:
#         filenames = f.readlines()

#         for file_base in filenames:
#             file_base = file_base.strip() # Remove newline characters

#             # Source paths
#             src_image_path = os.path.join(dataset_root, hsi_data_dir, file_base + '.npy')
#             src_mask_path = os.path.join(dataset_root, mask_data_dir, file_base + '.png')

#             # Destination paths
#             dest_image_path = os.path.join(dest_image_dir, file_base + '.npy')
#             dest_mask_path = os.path.join(dest_mask_dir, file_base + '.png')

#             if os.path.exists(src_image_path) and os.path.exists(src_mask_path):
#                 shutil.copy(src_image_path, dest_image_path)
#                 shutil.copy(src_mask_path, dest_mask_path)
#             else:
#                 print(f"Warning: Skipping {file_base} - Image or mask not found: {src_image_path}, {src_mask_path}")

# print("Dataset organization complete!")

Starting dataset organization...
Processing train split using /content/HSSDataset/HSS_VOC/split/train.txt
Processing val split using /content/HSSDataset/HSS_VOC/split/val.txt
Processing test split using /content/HSSDataset/HSS_VOC/split/test.txt
Dataset organization complete!


In [38]:
import tensorflow as tf
import os

class HSSSegmentationDatasetTF:
    """
    A TensorFlow Dataset class for loading segmentation datasets from the HSSDataset structure.
    Expected structure: destination_root/[train|val|test]/images/image.npy
    and destination_root/[train|val|test]/masks/image.png (corresponding mask)
    """
    def __init__(
        self,
        dataset_root="/content/HSSDataset_Splitted", # Now points to the organized data
        image_dir="images", # Subdirectory inside split
        mask_dir="masks",   # Subdirectory inside split
        image_size=(512, 512),
        batch_size=32,
        split="train",
        img_ext=".npy",
        mask_ext=".png",
        img_channels=25, # Default to 25 channels for HSSDataset
        mask_channels=1
    ):
        self.dataset_root = dataset_root # This is now the organized dataset root
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_size = image_size
        self.batch_size = batch_size
        self.image_paths = []
        self.mask_paths = []
        self.split = split
        self.img_ext = img_ext
        self.mask_ext = mask_ext
        self.img_channels = img_channels
        self.mask_channels = mask_channels
        self._load_paths()

    def _load_paths(self):
        print(f"Inspecting dataset structure for segmentation at: {self.dataset_root}/{self.split}")

        current_split_images_path = os.path.join(self.dataset_root, self.split, self.image_dir)
        current_split_masks_path = os.path.join(self.dataset_root, self.split, self.mask_dir)

        if not os.path.isdir(current_split_images_path) or not os.path.isdir(current_split_masks_path):
            print(f"Error: Image or mask directory not found for split '{self.split}' at {self.dataset_root}")
            return

        # List all image files in the split's image directory
        image_filenames = sorted([f for f in os.listdir(current_split_images_path) if f.lower().endswith(self.img_ext)])

        for img_filename in image_filenames:
            base_filename = os.path.splitext(img_filename)[0]
            mask_filename = base_filename + self.mask_ext

            img_filepath = os.path.join(current_split_images_path, img_filename)
            mask_filepath = os.path.join(current_split_masks_path, mask_filename)

            if os.path.exists(img_filepath) and os.path.exists(mask_filepath):
                self.image_paths.append(img_filepath)
                self.mask_paths.append(mask_filepath)
            else:
                print(f"Warning: Skipping {base_filename} - Corresponding mask not found: {mask_filepath}")

        if not self.image_paths:
            print(f"Warning: No image-mask pairs found in '{current_split_images_path}' for split '{self.split}'.")
        else:
            print(f"Found {len(self.image_paths)} image-mask pairs for split '{self.split}'.")

    def _load_npy(self, path):
      arr = np.load(path.decode("utf-8"))
      return arr.astype(np.float32)
    
    @tf.function
    def _parse_image_mask(self, image_path, mask_path):
        image = tf.numpy_function(self._load_npy, [image_path], tf.float32)
        image.set_shape([None, None, self.img_channels])

        mask = tf.io.read_file(mask_path)
        mask = tf.image.decode_png(mask, channels=1)

        mask = tf.cast(mask > 0, tf.int32)
        # mask = mask[:, :, 0:1] # Ensure mask has shape (H, W, 1)

        return image, mask

    def get_dataset(self, shuffle=False):
        if not self.image_paths:
            print("Dataset paths not loaded. Cannot create a TensorFlow dataset.")
            return None

        # Create a TensorFlow dataset from image and mask paths
        dataset = tf.data.Dataset.from_tensor_slices((self.image_paths, self.mask_paths))
        # dataset = dataset.interleave(
        #     lambda img, mask: tf.data.Dataset.from_tensors((img, mask)),
        #     cycle_length=8,          # number of parallel readers
        #     num_parallel_calls=tf.data.AUTOTUNE
        # )

        if shuffle:
            dataset = dataset.shuffle(buffer_size=len(self.image_paths)) # Shuffle the dataset

        # Map the parsing function to load and preprocess images and masks
        dataset = dataset.map(self._parse_image_mask, num_parallel_calls=tf.data.AUTOTUNE)

        # # Caching the dataset (consider if memory allows, especially for large datasets)
        # dataset = dataset.cache()

        dataset = dataset.batch(self.batch_size) # Batch the dataset
        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE) # Pre-fetch for performance
        return dataset


In [39]:

print("\n--- Loading HSSSegmentationDatasetTF ---")

# Loading training data
train_segmentation_data = HSSSegmentationDatasetTF(
    dataset_root="HSSDataset_organized", # New root for organized data
    image_dir="images",
    mask_dir="masks",
    image_size=(216, 408),
    batch_size=16,
    split="train",
    img_channels=25,
    mask_channels=1
)

tf_train_dataset = train_segmentation_data.get_dataset(shuffle=True)
print("Type of train dataset: ", type(tf_train_dataset))

# Loading validation data
val_segmentation_data = HSSSegmentationDatasetTF(
    dataset_root="HSSDataset_organized",
    image_dir="images",
    mask_dir="masks",
    image_size=(216, 408),
    batch_size=16,
    split="val",
    img_channels=25,
    mask_channels=1
)

tf_val_dataset = val_segmentation_data.get_dataset(shuffle=False)
print("Type of validation dataset: ", type(tf_val_dataset))

# Loading test data
test_segmentation_data = HSSSegmentationDatasetTF(
    dataset_root="HSSDataset_organized",
    image_dir="images",
    mask_dir="masks",
    image_size=(216, 408),
    batch_size=16,
    split="test",
    img_channels=25,
    mask_channels=1
)

tf_test_dataset = test_segmentation_data.get_dataset(shuffle=False)
print("Type of test dataset: ", type(tf_test_dataset))


if tf_train_dataset:
    for image_batch, mask_batch in tf_train_dataset.take(1):
        print(f"Shape of one image batch from train dataset: {image_batch.shape}") # (batch_size, image_height, image_width, channels)
        print(f"Shape of one mask batch from train dataset: {mask_batch.shape}") # (batch_size, mask_height, mask_width, 1)
        print(f"Example mask pixel values (first image in batch, top-left corner):\n {mask_batch[0, :5, :5, 0].numpy()}")
        break
else:
    print("Failed to create TensorFlow training dataset. Check warnings above.")


if tf_val_dataset:
    for image_batch, mask_batch in tf_val_dataset.take(1):
        print(f"Shape of one image batch from validation dataset: {image_batch.shape}") # (batch_size, image_height, image_width, channels)
        print(f"Shape of one mask batch from validation dataset: {mask_batch.shape}") # (batch_size, mask_height, mask_width, 1)
        break
else:
    print("Failed to create TensorFlow validation dataset. Check warnings above.")

if tf_test_dataset:
    for image_batch, mask_batch in tf_test_dataset.take(1):
        print(f"Shape of one image batch from test dataset: {image_batch.shape}") # (batch_size, image_height, image_width, channels)
        print(f"Shape of one mask batch from test dataset: {mask_batch.shape}") # (batch_size, mask_height, mask_width, 1)
        break
else:
    print("Failed to create TensorFlow test dataset. Check warnings above.")



--- Loading HSSSegmentationDatasetTF ---
Inspecting dataset structure for segmentation at: HSSDataset_organized/train
Found 604 image-mask pairs for split 'train'.
Type of train dataset:  <class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
Inspecting dataset structure for segmentation at: HSSDataset_organized/val
Found 201 image-mask pairs for split 'val'.
Type of validation dataset:  <class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
Inspecting dataset structure for segmentation at: HSSDataset_organized/test
Found 202 image-mask pairs for split 'test'.
Type of test dataset:  <class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
Shape of one image batch from train dataset: (16, 216, 408, 25)
Shape of one mask batch from train dataset: (16, 216, 408, 1)
Example mask pixel values (first image in batch, top-left corner):
 [[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Shape of one image batch from validation dataset: (16, 216, 4

In [41]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D,
    BatchNormalization, Activation, Concatenate
)
from tensorflow.keras.models import Model


def conv_block(x, filters, kernel_size=(3, 3)):
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x


def justo_unet_simple(input_shape=(256, 256, 1)):
    inputs = Input(shape=input_shape)

    # ---------------- Encoder ----------------
    # Level 1
    c1 = conv_block(inputs, 6)
    p1 = MaxPooling2D((2, 2))(c1)

    # Level 2 (bottleneck)
    c2 = conv_block(p1, 12)
    p2 = MaxPooling2D((2, 2))(c2)

    # ---------------- Bottleneck ----------------
    c3 = conv_block(p2, 6)

    # ---------------- Decoder ----------------
    # Up 1
    u1 = UpSampling2D((2, 2))(c3)
    u1 = Concatenate()([u1, c2])
    c4 = conv_block(u1, 12)

    # Up 2
    u2 = UpSampling2D((2, 2))(c4)
    u2 = Concatenate()([u2, c1])
    c5 = conv_block(u2, 6)

    # ---------------- Output ----------------
    outputs = Conv2D(1, (3, 3), padding='same', activation='sigmoid')(c5)

    model = Model(inputs, outputs, name="2D_Justo_UNet_Simple_Binary")

    return model


# Build model
model = justo_unet_simple(input_shape=(216, 408, 25))
model.summary()

Model: "2D_Justo_UNet_Simple_Binary"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14      │ (None, 216, 408,  │          0 │ -                 │
│ (InputLayer)        │ 25)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_82 (Conv2D)  │ (None, 216, 408,  │      1,356 │ input_layer_14[0… │
│                     │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 216, 408,  │         24 │ conv2d_82[0][0]   │
│ (BatchNormalizatio… │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_65       │ (None, 216, 408,  │          0 │ batch_normalizat… │
│ (Activation)        │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_27    │ (None, 108, 204,  │          0 │ activation_65[0]… │
│ (MaxPooling2D)      │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_83 (Conv2D)  │ (None, 108, 204,  │        660 │ max_pooling2d_27… │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 108, 204,  │         48 │ conv2d_83[0][0]   │
│ (BatchNormalizatio… │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_66       │ (None, 108, 204,  │          0 │ batch_normalizat… │
│ (Activation)        │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_28    │ (None, 54, 102,   │          0 │ activation_66[0]… │
│ (MaxPooling2D)      │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_84 (Conv2D)  │ (None, 54, 102,   │        654 │ max_pooling2d_28… │
│                     │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 102,   │         24 │ conv2d_84[0][0]   │
│ (BatchNormalizatio… │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_67       │ (None, 54, 102,   │          0 │ batch_normalizat… │
│ (Activation)        │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_28    │ (None, 108, 204,  │          0 │ activation_67[0]… │
│ (UpSampling2D)      │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_26      │ (None, 108, 204,  │          0 │ up_sampling2d_28… │
│ (Concatenate)       │ 18)               │            │ activation_66[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_85 (Conv2D)  │ (None, 108, 204,  │      1,956 │ concatenate_26[0… │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 108, 204,  │         48 │ conv2d_85[0][0]   │
│ (BatchNormalizatio… │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_68       │ (None, 108, 204,  │          0 │ batch_normalizat

 Total params: 5,827 (22.76 KB)

 Trainable params: 5,743 (22.43 KB)

 Non-trainable params: 84 (336.00 B)

In [42]:
import os
import tensorflow as tf
from tensorflow import keras
from keras import callbacks


class HSSDataLoader:
    def __init__(self, data_path, image_size, batch_size, split, img_channels, mask_channels):
        self.data_path = data_path
        self.image_size = image_size
        self.batch_size = batch_size
        self.split = split
        self.img_channels = img_channels
        self.mask_channels = mask_channels

    def build_dataset(self, shuffle=True):
        dataset_loader = HSSSegmentationDatasetTF(
            dataset_root=self.data_path,
            image_dir="images",
            mask_dir="masks",
            image_size=self.image_size,
            batch_size=self.batch_size,
            split=self.split,
            img_channels=self.img_channels,
            mask_channels=self.mask_channels
        )
        return dataset_loader.get_dataset(shuffle=shuffle)


class SegmentationTrainer:
    """
    Trainer class for UNet segmentation model.

    Handles model creation, compilation, training, and evaluation.
    """
    def __init__(self, train_path, val_path, test_path, input_shape, num_classes, base_filters, learning_rate, batch_size):
        """
        Initialize the trainer.

        Args:
            train_path: Path to training dataset (e.g., /content/HSSDataset_Splitted)
            val_path: Path to validation dataset
            test_path: Path to test dataset
            input_shape: Input image shape (height, width, channels)
            num_classes: Number of segmentation classes
            base_filters: Base number of filters in UNet
            learning_rate: Learning rate for optimizer
            batch_size: Batch size for training
        """
        self.train_path = train_path
        self.val_path = val_path
        self.test_path = test_path
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.base_filters = base_filters
        self.learning_rate = learning_rate
        self.batch_size = batch_size

        # Build model
        self.model = None
        self.history = None

    def build_model(self):
        """Build and compile the UNet model"""
        print("Building UNet model...")
        self.model = justo_unet_simple(input_shape=self.input_shape)

        # Explicitly build the model with a dummy input to handle the Resizing layer
        # This ensures the model knows its input shape before training
        dummy_input = tf.ones((1,) + self.input_shape, dtype=tf.float32)
        _ = self.model(dummy_input, training=False)

        # Compile model with appropriate loss for multi-class segmentation
        # Use SparseCategoricalCrossentropy for integer labels
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss=keras.losses.BinaryCrossentropy(from_logits=False),
            metrics=['accuracy',
                    keras.metrics.MeanIoU(
                        num_classes=self.num_classes,
                        sparse_y_true=True,
                        sparse_y_pred=False
                    )
            ]
        )

        print("Model compiled successfully!")
        return self.model

    def prepare_datasets(self):
        """Create and prepare training and validation datasets"""
        print(f"Loading training data from: {self.train_path}")
        train_loader = HSSDataLoader(
            data_path=self.train_path,
            image_size=self.input_shape[:2],
            batch_size=self.batch_size,
            split="train",
            img_channels=self.input_shape[-1],
            mask_channels=1 # Assuming 1 channel for masks
        )
        train_dataset = train_loader.build_dataset(shuffle=True)

        print(f"Loading validation data from: {self.val_path}")
        val_loader = HSSDataLoader(
            data_path=self.val_path,
            image_size=self.input_shape[:2],
            batch_size=self.batch_size,
            split="val",
            img_channels=self.input_shape[-1],
            mask_channels=1 # Assuming 1 channel for masks
        )
        val_dataset = val_loader.build_dataset(shuffle=False)

        return train_dataset, val_dataset

    def train(self, epochs=50, save_path="../models"):
        """
        Train the model.

        Args:
            epochs: Number of training epochs
            save_path: Path to save model checkpoints

        Returns:
            Training history
        """
        # Create save directory
        os.makedirs(save_path, exist_ok=True)

        # Build and compile model
        self.build_model()

        assert self.model is not None, "Model building failed. Cannot train."

        # Print model summary
        self.model.summary()

        # Prepare datasets
        train_dataset, val_dataset = self.prepare_datasets()

        # Define callbacks
        callback_list = [
            # Save best model based on validation loss
            callbacks.ModelCheckpoint(
                filepath=os.path.join(save_path, "best_model.keras"),
                monitor='val_mean_io_u',
                save_best_only=True,
                mode='max',
                verbose=1
            ),
            # Early stopping
            callbacks.EarlyStopping(
                monitor='val_mean_io_u',
                patience=10,
                restore_best_weights=True,
                mode='max',
                verbose=1
            ),
            # Reduce learning rate on plateau
            callbacks.ReduceLROnPlateau(
                monitor='val_mean_io_u',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                mode='max',
                verbose=1
            ),
            # TensorBoard logging
            callbacks.TensorBoard(
                log_dir=os.path.join(save_path, "logs_1")
            )
        ]

        print("\n" + "="*50)
        print("Starting training...")
        print(f"Epochs: {epochs}")
        print(f"Batch size: {self.batch_size}")
        print(f"Learning rate: {self.learning_rate}")
        print("="*50 + "\n")

        # Train the model
        self.history = self.model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=epochs,
            callbacks=callback_list,
            verbose="auto"
        )

        # Save final model
        final_model_path = os.path.join(save_path, "final_model.keras")
        self.model.save(final_model_path)
        print(f"\nFinal model saved to: {final_model_path}")

        return self.history

    def evaluate(self):
        """
        Evaluate the model on test data.

        Returns:
            Evaluation metrics
        """
        if self.model is None:
            print("No model loaded. Building model...")
            self.build_model()

        assert self.model is not None, "Model building failed. Cannot evaluate."

        print(f"\nEvaluating on test data from: {self.test_path}")
        test_loader = HSSDataLoader(
            data_path=self.test_path,
            image_size=self.input_shape[:2],
            batch_size=self.batch_size,
            split="test",
            img_channels=self.input_shape[-1],
            mask_channels=1 # Assuming 1 channel for masks
        )
        test_dataset = test_loader.build_dataset(shuffle=False)

        results = self.model.evaluate(test_dataset, verbose=1)

        print("\nTest Results:")
        print(f"  Loss: {results[0]:.4f}")
        print(f"  Accuracy: {results[1]:.4f}")
        print(f"  Mean IoU: {results[2]:.4f}")

        return results

    def predict(self, image):
        """
        Make predictions on a single image.

        Args:
            image: Input image tensor (H, W, C)

        Returns:
            Predicted segmentation mask
        """
        if self.model is None:
            raise ValueError("Model not built. Call build_model() first.")

        # Add batch dimension
        image = tf.expand_dims(image, axis=0)

        # Predict
        prediction = self.model.predict(image, verbose=0)

        # Get class with highest probability
        segmentation_map = tf.argmax(prediction, axis=-1)

        return segmentation_map[0].numpy()


def main():
    """Main training function"""
    # Configuration
    CONFIG = {
        'train_path': 'HSSDataset_organized', # Updated path
        'val_path': 'HSSDataset_organized',   # Updated path
        'test_path': 'HSSDataset_organized',  # New test_path
        'input_shape': (216, 408, 25),  # Example shape, adjust as needed
        'num_classes': 1, # binary segmentation (background + object)
        'base_filters': 16, # Reduced base filters for quicker testing
        'learning_rate': 1e-4,
        'batch_size': 8, # Reduced batch size for quicker testing
        'epochs': 200, # Reduced epochs for quicker testing
        'save_path': './models'
    }

    print("HSI Segmentation Training")
    print("="*50)
    print(f"Configuration:")
    for key, value in CONFIG.items():
        print(f"  {key}: {value}")
    print("="*50 + "\n")

    # Create trainer
    trainer = SegmentationTrainer(
        train_path=CONFIG['train_path'],
        val_path=CONFIG['val_path'],
        test_path=CONFIG['test_path'],
        input_shape=CONFIG['input_shape'],
        num_classes=CONFIG['num_classes'],
        base_filters=CONFIG['base_filters'],
        learning_rate=CONFIG['learning_rate'],
        batch_size=CONFIG['batch_size']
    )

    # Train the model
    history = trainer.train(
        epochs=CONFIG['epochs'],
        save_path=CONFIG['save_path']
    )

    # Evaluate on test set
    trainer.evaluate()

    print("\nTraining completed!")

    return history

history = main()

HSI Segmentation Training
Configuration:
  train_path: HSSDataset_organized
  val_path: HSSDataset_organized
  test_path: HSSDataset_organized
  input_shape: (216, 408, 25)
  num_classes: 1
  base_filters: 16
  learning_rate: 0.0001
  batch_size: 8
  epochs: 200
  save_path: ./models

Building UNet model...
Model compiled successfully!


Model: "2D_Justo_UNet_Simple_Binary"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_15      │ (None, 216, 408,  │          0 │ -                 │
│ (InputLayer)        │ 25)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_88 (Conv2D)  │ (None, 216, 408,  │      1,356 │ input_layer_15[0… │
│                     │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 216, 408,  │         24 │ conv2d_88[0][0]   │
│ (BatchNormalizatio… │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_70       │ (None, 216, 408,  │          0 │ batch_normalizat… │
│ (Activation)        │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_29    │ (None, 108, 204,  │          0 │ activation_70[0]… │
│ (MaxPooling2D)      │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_89 (Conv2D)  │ (None, 108, 204,  │        660 │ max_pooling2d_29… │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 108, 204,  │         48 │ conv2d_89[0][0]   │
│ (BatchNormalizatio… │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_71       │ (None, 108, 204,  │          0 │ batch_normalizat… │
│ (Activation)        │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_30    │ (None, 54, 102,   │          0 │ activation_71[0]… │
│ (MaxPooling2D)      │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_90 (Conv2D)  │ (None, 54, 102,   │        654 │ max_pooling2d_30… │
│                     │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 102,   │         24 │ conv2d_90[0][0]   │
│ (BatchNormalizatio… │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_72       │ (None, 54, 102,   │          0 │ batch_normalizat… │
│ (Activation)        │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_30    │ (None, 108, 204,  │          0 │ activation_72[0]… │
│ (UpSampling2D)      │ 6)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_28      │ (None, 108, 204,  │          0 │ up_sampling2d_30… │
│ (Concatenate)       │ 18)               │            │ activation_71[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_91 (Conv2D)  │ (None, 108, 204,  │      1,956 │ concatenate_28[0… │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 108, 204,  │         48 │ conv2d_91[0][0]   │
│ (BatchNormalizatio… │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_73       │ (None, 108, 204,  │          0 │ batch_normalizat

 Total params: 5,827 (22.76 KB)

 Trainable params: 5,743 (22.43 KB)

 Non-trainable params: 84 (336.00 B)

Loading training data from: HSSDataset_organized
Inspecting dataset structure for segmentation at: HSSDataset_organized/train
Found 604 image-mask pairs for split 'train'.
Loading validation data from: HSSDataset_organized
Inspecting dataset structure for segmentation at: HSSDataset_organized/val
Found 201 image-mask pairs for split 'val'.

Starting training...
Epochs: 200
Batch size: 8
Learning rate: 0.0001

Epoch 1/200


InvalidArgumentError: Graph execution error:

Detected at node ScatterNd defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\asyncio\base_events.py", line 645, in run_forever

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\asyncio\base_events.py", line 1999, in _run_once

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\asyncio\events.py", line 88, in _run

  File "c:\Users\rajba\miniconda3\envs\uwl\Lib\site-pa

In [ ]:
import matplotlib.pyplot as plt

def plot_metrics(history):
    """Plots training and validation metrics from a Keras history object."""
    metrics = ['loss', 'accuracy', 'mean_io_u'] # Include mean_io_u
    for metric in metrics:
        plt.figure(figsize=(10, 6))
        plt.plot(history.history[metric], label=f'Training {metric}')
        val_metric = f'val_{metric}'
        if val_metric in history.history:
            plt.plot(history.history[val_metric], label=f'Validation {metric}')
        plt.title(f'Training and Validation {metric.replace("_", " ").title()}')
        plt.xlabel('Epoch')
        plt.ylabel(metric.replace("_", " ").title())
        plt.legend()
        plt.grid(True)
        plt.show()

# Assuming 'history' object is available from the previous training run
if 'history' in locals() and history is not None:
    print("Plotting training history...")
    plot_metrics(history)
else:
    print("Error: Training history not found. Please ensure the model has been trained.")


Error: Training history not found. Please ensure the model has been trained.
